# judgeaudit on Google Colab

Audit an LLM judge, put a guard around it, and get the cheapest set-up you can trust,
on a free Colab GPU.

**Before you start:** Runtime > Change runtime type > **T4 GPU** (any NVIDIA GPU works).

What this notebook does:

1. Installs judgeaudit and runs its tests (no GPU needed).
2. Builds llama.cpp's `llama-server` with CUDA (about 5 to 10 minutes, once per session).
3. Downloads Llama 3.1 8B Instruct (4-bit, 4.9 GB) and audits it in 8 set-ups:
   plain and guarded prompts, with and without a reference answer, with and without
   the guard. This is the quick version of the full experiment.
4. Shows the recommendation and the interactive dashboard.

Expect roughly 20 to 40 minutes on a T4 for step 3. Set `LIMIT_ITEMS = 20` below for a
5 to 10 minute demo on fewer questions.

In [ ]:
!nvidia-smi

## 1. Install judgeaudit and run its tests

In [ ]:
!git clone -q https://github.com/aarsh-adhvaryu/judgeaudit.git /content/judgeaudit
%cd /content/judgeaudit
!pip install -q -e ".[dev]"
!python -m pytest -q

## 2. Build llama-server with CUDA

Pinned to llama.cpp release `b12752`, the build the published results came from.
The GPU's compute capability is detected so only its kernels are compiled.

In [ ]:
import subprocess
cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.split()
ARCH = cap[0].replace(".", "") if cap else "75"
print("CUDA architecture:", ARCH)
!git clone -q --depth 1 --branch b12752 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={ARCH} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /content/cmake.log 2>&1 || tail -30 /content/cmake.log
!cmake --build /content/llama.cpp/build --config Release -j 4 --target llama-server > /content/build.log 2>&1 || tail -30 /content/build.log
!ls -la /content/llama.cpp/build/bin/llama-server

## 3. Run the audit

`--quick` runs 8 set-ups on one model. Drop `--quick` for all 44 set-ups, and add
`--models llama31-8b-q4,qwen3-8b-q4` (or any of `llama32-3b-q4`, `llama31-8b-q8`,
`qwen3-8b-q8`) to compare models. The verdict cache makes a re-run after a disconnect
resume where it stopped, as long as the session's files are still there.

In [ ]:
LIMIT_ITEMS = None   # e.g. 20 for a short demo; None = all 100 questions
SERVER = "/content/llama.cpp/build/bin/llama-server"
limit = f"--limit-items {LIMIT_ITEMS}" if LIMIT_ITEMS else ""
!python scripts/run_grid.py --server {SERVER} --models llama31-8b-q4 --quick {limit}

## 4. Results

The recommendation chooses on the dev half of the suite and reports the held-out half.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/recommendation.md", encoding="utf-8").read()))

The dashboard: the recommendation, a trust-against-cost chart, every set-up, and a
browser of the answers that fooled each judge. Shown below, and downloadable.

In [ ]:
from IPython.display import HTML, display
display(HTML(open("results/dashboard.html", encoding="utf-8").read()))

In [ ]:
from google.colab import files
files.download("results/dashboard.html")

## Optional: no GPU

The same pipeline against two mock judges with planted biases, in about a minute.
Results go to `results-mock/`.

In [ ]:
!python scripts/run_grid.py --mock --quick